# Análise Exploratória (EDA) - Abstenção ENEM 2023

Neste notebook, vamos explorar uma amostra do dataset gerado pelo nosso pipeline de processamento em massa (`src/process_real_enem.py`) para investigar visualmente os fatores que influenciam a abstenção no Exame Nacional do Ensino Médio.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Configuração visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Carregando os Dados Consolidados
Vamos ler o arquivo JSON exportado pelo nosso pipeline de Big Data.

In [ ]:
with open('../data/enem_metrics_final.json', 'r', encoding='utf-8') as f:
    db = json.load(f)

print(f"Dados carregados para Brasil + {len(db.keys()) - 1} UFs.")

## 2. Análise Nacional: Renda x Abstenção
Vamos plotar como a renda afeta diretamente a presença dos candidatos.

In [ ]:
# Extraindo dados do Brasil
df_renda = pd.DataFrame(db['BR']['renda'])

plt.figure(figsize=(10, 5))
ax = sns.barplot(x='faixa', y='taxa', data=df_renda, palette='viridis')
plt.title('Taxa de Abstenção por Faixa de Renda (Nacional)', fontsize=14, pad=15)
plt.ylabel('Taxa de Abstenção (%)')
plt.xlabel('Renda Familiar')

# Adicionando rótulos
for i in ax.containers:
    ax.bar_label(i, fmt='%.1f%%', padding=3)
    
plt.show()

**Insight:** Fica evidente a correlação negativa entre renda e abstenção. Candidatos sem renda faltam massivamente mais do que candidatos de classes mais altas.

## 3. Disparidade Regional (Abstenção por UF)

In [ ]:
ufs = []
taxas = []

for uf, data in db.items():
    if uf != 'BR':
        ufs.append(uf)
        taxas.append(data['kpis']['taxa_abstencao'])

df_ufs = pd.DataFrame({'UF': ufs, 'Taxa': taxas})
df_ufs = df_ufs.sort_values(by='Taxa', ascending=False)

plt.figure(figsize=(14, 6))
ax = sns.barplot(x='UF', y='Taxa', data=df_ufs, palette='coolwarm')
plt.title('Taxa de Abstenção por Estado', fontsize=14, pad=15)
plt.ylabel('Taxa de Abstenção (%)')
plt.xlabel('Unidade Federativa')
plt.xticks(rotation=45)
plt.axhline(y=db['BR']['kpis']['taxa_abstencao'], color='r', linestyle='--', label='Média Nacional')
plt.legend()
plt.tight_layout()
plt.show()

**Insight:** O mapa de calor em barra mostra que estados do Norte (ex: AM) têm uma taxa de abstenção muito superior à média nacional, enquanto estados do Sul/Sudeste (ex: SP, RS) apresentam as menores taxas.